# Prediction of Cases

In [ ]:
import pandas as pd
#import matplot
import numpy as np

import matplotlib as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
merged = pd.read_csv('/content/drive/MyDrive/Thesis Data/merged.csv')
merged.head()


,BARANGAY,YEAR,MONTH,Cases,RAINFALL,TMAX,TMIN,WIND_SPEED,WIND_DIRECTION,RH,TAVG
0,BUROL,2010,1,0,5.8,30.284615,24.661538,4.0,110.0,76.384615,27.473077
1,BUROL,2011,1,10,69.7,30.000000,24.700000,3.0,110.0,76.000000,27.350000
2,BUROL,2012,1,0,25.2,30.800000,25.300000,3.0,140.0,78.000000,28.050000
3,BUROL,2013,1,2,37.6,30.200000,24.200000,4.0,140.0,78.000000,27.200000
4,BUROL,2014,1,0,0.4,29.500000,23.400000,3.0,140.0,72.000000,26.450000


In [ ]:
merged = merged.sort_values(by=['BARANGAY', 'YEAR', 'MONTH']).reset_index(drop=True)

merged['Cases_Lagged'] = merged.groupby('BARANGAY')['Cases'].shift(1)

merged['Cases_Rolling_Avg_3M'] = merged.groupby('BARANGAY')['Cases'].rolling(window=3).mean().reset_index(level=0, drop=True)



In [ ]:
merged['Cases_Lagged'] = merged['Cases_Lagged'].fillna(0)
merged['Cases_Rolling_Avg_3M'] = merged['Cases_Rolling_Avg_3M'].fillna(0)

merged.head()

,BARANGAY,YEAR,MONTH,Cases,RAINFALL,TMAX,TMIN,WIND_SPEED,WIND_DIRECTION,RH,TAVG,Cases_Lagged,Cases_Rolling_Avg_3M
0,BUROL,2010,1,0,5.8,30.284615,24.661538,4.0,110.0,76.384615,27.473077,0.0,0.0
1,BUROL,2010,2,0,0.0,31.169231,24.776923,4.0,110.0,74.615385,27.973077,0.0,0.0
2,BUROL,2010,3,0,10.4,32.569231,25.800000,4.0,110.0,72.846154,29.184615,0.0,0.0
3,BUROL,2010,4,0,45.4,34.300000,27.030769,5.0,110.0,70.769231,30.665385,0.0,0.0
4,BUROL,2010,5,0,36.8,34.892308,27.876923,4.0,110.0,73.615385,31.384615,0.0,0.0


In [ ]:
X = merged.drop(['BARANGAY', 'Cases'], axis=1)
y = merged['Cases']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

display(X_train.head())
display(y_train.head())

,YEAR,MONTH,RAINFALL,TMAX,TMIN,WIND_SPEED,WIND_DIRECTION,RH,TAVG,Cases_Lagged,Cases_Rolling_Avg_3M
2688,2014,1,0.4,29.5,23.4,3.0,140.0,72.0,26.45,2.0,2.666667
10610,2018,3,105.8,32.1,25.7,3.0,90.0,75.0,28.90,0.0,0.666667
3400,2022,5,232.2,34.1,27.1,3.0,90.0,75.0,30.60,0.0,0.333333
2899,2017,8,323.0,32.6,27.1,3.0,220.0,80.0,29.85,2.0,0.666667
4894,2010,11,240.9,31.3,26.2,3.0,110.0,82.0,28.75,2.0,2.000000


,Cases
2688,3
10610,0
3400,1
2899,0
4894,2


In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import numpy as np

# Initialize the XGBoost Regressor
xgb_reg = xgb.XGBRegressor(
    objective='reg:squarederror',  # regression objective
    n_estimators=500,              # number of boosting rounds
    learning_rate=0.05,            # step size shrinkage
    max_depth=6,                   # tree depth
    subsample=0.8,                 # row sampling
    colsample_bytree=0.8,          # feature sampling
    random_state=42,
    n_jobs=-1
)

# Train the model
xgb_reg.fit(X_train, y_train)

# Predictions
y_pred_xgb = xgb_reg.predict(X_test)

# Evaluate
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mse_xgb)

print(f"XGBoost RMSE on Test Set: {rmse_xgb}")

XGBoost RMSE on Test Set: 1.409868374761354


# Risk Classification

In [ ]:
# Apply threshold

RISK_THRESHOLDS = {
    'low_max': 5,      # 0–5 → Low
    'medium_max': 10    # 11–30 → Medium, 31+ → High
}

def assign_risk_level(predicted_cases):
    if predicted_cases <= RISK_THRESHOLDS['low_max']:
        return 'Low'
    elif predicted_cases <= RISK_THRESHOLDS['medium_max']:
        return 'Medium'
    else:
        return 'High'

In [ ]:
import pandas as pd
import numpy as np

# Load and sort data
merged = pd.read_csv('/content/drive/MyDrive/Thesis Data/merged.csv')
merged['DATE'] = pd.to_datetime(merged[['YEAR', 'MONTH']].assign(DAY=1))
merged = merged.sort_values(['BARANGAY', 'DATE']).reset_index(drop=True)

# Create time-based features
merged['Cases_Lagged'] = merged.groupby('BARANGAY')['Cases'].shift(1)
merged['Cases_Rolling_Avg_3M'] = merged.groupby('BARANGAY')['Cases'].rolling(3).mean().reset_index(level=0, drop=True)

# Fill NaNs (common for first months)
merged['Cases_Lagged'] = merged['Cases_Lagged'].fillna(0)
merged['Cases_Rolling_Avg_3M'] = merged['Cases_Rolling_Avg_3M'].fillna(0)

In [ ]:
# Example: Train up to end of 2022, test on 2023
train_data = merged[merged['DATE'] < '2023-01-01']
test_data = merged[merged['DATE'] >= '2023-01-01']

# Features and target
feature_cols = [col for col in merged.columns if col not in ['BARANGAY', 'Cases', 'DATE']]
X_train = train_data[feature_cols]
y_train = train_data['Cases']
X_test = test_data[feature_cols]
y_test = test_data['Cases']

In [ ]:
import xgboost as xgb

xgb_reg = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_reg.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=-1, num_parallel_tree=None, ...)

In [ ]:
# Predict case counts
y_pred_cases = xgb_reg.predict(X_test)

# Convert to risk levels
y_pred_risk = [assign_risk_level(c) for c in y_pred_cases]
y_true_risk = [assign_risk_level(c) for c in y_test]

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse = np.sqrt(mean_squared_error(y_test, y_pred_cases))
mae = mean_absolute_error(y_test, y_pred_cases)
print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}")

RMSE: 0.76, MAE: 0.43


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print("Risk Classification Report:")
print(classification_report(y_true_risk, y_pred_risk))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true_risk, y_pred_risk))

Risk Classification Report:
              precision    recall  f1-score   support

        High       0.00      0.00      0.00         1
         Low       1.00      0.99      1.00       786
      Medium       0.25      0.40      0.31         5

    accuracy                           0.99       792
   macro avg       0.42      0.46      0.43       792
weighted avg       0.99      0.99      0.99       792


Confusion Matrix:
[[  0   0   1]
 [  0 781   5]
 [  1   2   2]]


In [ ]:
results = pd.DataFrame({
    'BARANGAY': test_data['BARANGAY'].values,
    'YEAR': test_data['YEAR'].values,
    'MONTH': test_data['MONTH'].values,
    'Actual_Cases': y_test.values,
    'Predicted_Cases': y_pred_cases,
    'Actual_Risk': y_true_risk,
    'Predicted_Risk': y_pred_risk
})

# Save or display
results.to_csv('dengue_risk_predictions_2023.csv', index=False)
print(results.tail())

    BARANGAY  YEAR  MONTH  Actual_Cases  Predicted_Cases Actual_Risk  \
787  ZONE IV  2023      8             0         1.296105         Low   
788  ZONE IV  2023      9             0         0.691474         Low   
789  ZONE IV  2023     10             0         0.000612         Low   
790  ZONE IV  2023     11             0        -0.121812         Low   
791  ZONE IV  2023     12             0        -0.065165         Low   

    Predicted_Risk  
787            Low  
788            Low  
789            Low  
790            Low  
791            Low  


In [ ]:
#show cases with medium cases
medium_cases = results[results['Predicted_Risk'] == 'Medium']
print(medium_cases)

          BARANGAY  YEAR  MONTH  Actual_Cases  Predicted_Cases Actual_Risk  \
188  PALIPARAN III  2023      9             4         5.438477         Low   
221        SALAWAG  2023      6             5         5.636809         Low   
222        SALAWAG  2023      7             3         5.855154         Low   
223        SALAWAG  2023      8             3         6.684897         Low   
224        SALAWAG  2023      9            10         8.369754      Medium   
225        SALAWAG  2023     10            16         9.440302        High   
227        SALAWAG  2023     12             5         6.697642         Low   
597      SAN MATEO  2023     10             8         5.475388      Medium   

    Predicted_Risk  
188         Medium  
221         Medium  
222         Medium  
223         Medium  
224         Medium  
225         Medium  
227         Medium  
597         Medium  


Yearly

In [ ]:
monthly_results = pd.DataFrame({
    'BARANGAY': test_data['BARANGAY'].values,
    'YEAR': test_data['YEAR'].values,
    'MONTH': test_data['MONTH'].values,
    'Actual_Cases': y_test.values,
    'Predicted_Cases': y_pred_cases,  # from XGBoost Regressor
})


yearly = monthly_results.groupby(['BARANGAY', 'YEAR']).agg(
    Actual_Cases_Yearly=('Actual_Cases', 'sum'),
    Predicted_Cases_Yearly=('Predicted_Cases', 'sum'),
    Max_Monthly_Actual=('Actual_Cases', 'max'),
    Max_Monthly_Predicted=('Predicted_Cases', 'max')
).reset_index()




In [ ]:
# Example: Use 75th and 90th percentiles of historical yearly totals
historical_yearly = merged.groupby(['BARANGAY', 'YEAR'])['Cases'].sum().reset_index()
low_thresh = historical_yearly['Cases'].quantile(0.50)   # median
med_thresh = historical_yearly['Cases'].quantile(0.75)
high_thresh = historical_yearly['Cases'].quantile(0.90)

print(f"Yearly thresholds → Low≤{low_thresh:.0f}, Med≤{med_thresh:.0f}, High>{high_thresh:.0f}")

Yearly thresholds → Low≤6, Med≤16, High>35


In [ ]:
# Yearly risk thresholds (total cases per barangay per year)
# Low:     0 – 16
# Medium: 17 – 35
# High:   36+
YEARLY_THRESHOLDS = {
    'low_max': 16,       # inclusive
    'medium_max': 35     # inclusive
}

def yearly_risk_total(cases):
    if cases <= YEARLY_THRESHOLDS['low_max']:
        return 'Low'
    elif cases <= YEARLY_THRESHOLDS['medium_max']:
        return 'Medium'
    else:
        return 'High'

In [ ]:
risk_order = {'Low': 0, 'Medium': 1, 'High': 2}

yearly_peak = monthly_results.groupby(['BARANGAY', 'YEAR'])['Monthly_Risk'].apply(
    lambda risks: max(risks, key=lambda x: risk_order[x])
).reset_index(name='Annual_Risk_Peak')

In [ ]:
# Add total-based risk
yearly['Annual_Risk_Total'] = yearly['Predicted_Cases_Yearly'].apply(yearly_risk_total)

# Merge with peak-based risk (if using)
yearly = yearly.merge(yearly_peak, on=['BARANGAY', 'YEAR'], how='left')

# Final yearly report
yearly_report = yearly[['BARANGAY', 'YEAR',
                        'Actual_Cases_Yearly', 'Predicted_Cases_Yearly',
                        'Annual_Risk_Total', 'Annual_Risk_Peak',
                        'Max_Monthly_Predicted']]

In [ ]:
yearly_report.tail(10)

,BARANGAY,YEAR,Actual_Cases_Yearly,Predicted_Cases_Yearly,Annual_Risk_Total,Annual_Risk_Peak,Max_Monthly_Predicted
56,SANTA FE,2023,10,9.493864,Low,Low,2.258482
57,SANTA LUCIA (SAN JUAN II),2023,1,2.275315,Low,Low,1.410982
58,SANTA MARIA (BARANGAY 20),2023,1,1.948101,Low,Low,1.410982
59,SANTO CRISTO (BARANGAY III),2023,1,1.948101,Low,Low,1.410982
60,SANTO NIÑO I,2023,1,2.565433,Low,Low,1.410982
61,SANTO NIÑO II,2023,1,2.824836,Low,Low,1.410982
62,VICTORIA REYES,2023,2,2.723833,Low,Low,1.410982
63,ZONE I-B,2023,4,5.612479,Low,Low,1.972870
64,ZONE III,2023,1,1.948101,Low,Low,1.410982
65,ZONE IV,2023,1,2.740491,Low,Low,1.296105


In [ ]:
#check medium for yearly_report
medium_cases = yearly_report[yearly_report['Annual_Risk_Total'] == 'High']
medium_cases

,BARANGAY,YEAR,Actual_Cases_Yearly,Predicted_Cases_Yearly,Annual_Risk_Total,Annual_Risk_Peak,Max_Monthly_Predicted
18,SALAWAG,2023,70,68.273224,High,High,14.288764


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, classification_report, confusion_matrix

# Load and sort data
merged = pd.read_csv('/content/drive/MyDrive/Thesis Data/merged.csv')
merged['DATE'] = pd.to_datetime(merged[['YEAR', 'MONTH']].assign(DAY=1))
merged = merged.sort_values(['BARANGAY', 'DATE']).reset_index(drop=True)

# Create time-based features
merged['Cases_Lagged'] = merged.groupby('BARANGAY')['Cases'].shift(1)
merged['Cases_Rolling_Avg_3M'] = merged.groupby('BARANGAY')['Cases'].rolling(3).mean().reset_index(level=0, drop=True)

# Fill NaNs (common for first months)
merged['Cases_Lagged'] = merged['Cases_Lagged'].fillna(0)
merged['Cases_Rolling_Avg_3M'] = merged['Cases_Rolling_Avg_3M'].fillna(0)

# Example: Train up to end of 2022, test on 2023
train_data = merged[merged['DATE'] < '2023-01-01']
test_data = merged[merged['DATE'] >= '2023-01-01']

# Features and target
feature_cols = [col for col in merged.columns if col not in ['BARANGAY', 'Cases', 'DATE']]
X_train = train_data[feature_cols]
y_train = train_data['Cases']
X_test = test_data[feature_cols]
y_test = test_data['Cases']

# Initialize and train the XGBoost Regressor
xgb_reg = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_reg.fit(X_train, y_train)

# Predict case counts
y_pred_cases = np.clip(xgb_reg.predict(X_test), 0, None)

# Define monthly risk thresholds and assignment function
RISK_THRESHOLDS = {'low_max': 5, 'medium_max': 10}
# Low: ≤5, Medium: 6–10, High: ≥11

def assign_risk_level(predicted_cases):
    if predicted_cases <= RISK_THRESHOLDS['low_max']:
        return 'Low'
    elif predicted_cases <= RISK_THRESHOLDS['medium_max']:
        return 'Medium'
    else:
        return 'High'

# Convert to monthly risk levels
y_pred_risk = [assign_risk_level(c) for c in y_pred_cases]
y_true_risk = [assign_risk_level(c) for c in y_test]

# Create monthly results DataFrame
results = pd.DataFrame({
    'BARANGAY': test_data['BARANGAY'].values,
    'YEAR': test_data['YEAR'].values,
    'MONTH': test_data['MONTH'].values,
    'Actual_Cases': y_test.values,
    'Predicted_Cases': y_pred_cases,
    'Actual_Risk': y_true_risk,
    'Predicted_Risk': y_pred_risk
})

# Calculate yearly results
monthly_results = pd.DataFrame({
    'BARANGAY': test_data['BARANGAY'].values,
    'YEAR': test_data['YEAR'].values,
    'MONTH': test_data['MONTH'].values,
    'Actual_Cases': y_test.values,
    'Predicted_Cases': y_pred_cases,  # from XGBoost Regressor
    'Monthly_Risk': y_pred_risk # Use predicted monthly risk for yearly peak
})

yearly = monthly_results.groupby(['BARANGAY', 'YEAR']).agg(
    Actual_Cases_Yearly=('Actual_Cases', 'sum'),
    Predicted_Cases_Yearly=('Predicted_Cases', 'sum'),
    Max_Monthly_Actual=('Actual_Cases', 'max'),
    Max_Monthly_Predicted=('Predicted_Cases', 'max')
).reset_index()

# Define yearly risk thresholds and assignment function
YEARLY_THRESHOLDS = {
    'low_max': 16,       # inclusive
    'medium_max': 35     # inclusive
}

def yearly_risk_total(cases):
    if cases <= YEARLY_THRESHOLDS['low_max']:
        return 'Low'
    elif cases <= YEARLY_THRESHOLDS['medium_max']:
        return 'Medium'
    else:
        return 'High'

# Add total-based yearly risk
yearly['Annual_Risk_Total'] = yearly['Predicted_Cases_Yearly'].apply(yearly_risk_total)

# Determine yearly peak risk based on monthly predicted risk
risk_order = {'Low': 0, 'Medium': 1, 'High': 2}
yearly_peak = monthly_results.groupby(['BARANGAY', 'YEAR'])['Monthly_Risk'].apply(
    lambda risks: max(risks, key=lambda x: risk_order[x])
).reset_index(name='Annual_Risk_Peak')

# Merge with peak-based risk
yearly = yearly.merge(yearly_peak, on=['BARANGAY', 'YEAR'], how='left')

# Final yearly report
yearly_report = yearly[['BARANGAY', 'YEAR',
                        'Actual_Cases_Yearly', 'Predicted_Cases_Yearly',
                        'Annual_Risk_Total', 'Annual_Risk_Peak',
                        'Max_Monthly_Predicted']]

# Display results and yearly_report
print("Monthly Results (tail):")
display(results.tail())

print("\nYearly Report (tail):")
display(yearly_report.tail())

Monthly Results (tail):


,BARANGAY,YEAR,MONTH,Actual_Cases,Predicted_Cases,Actual_Risk,Predicted_Risk
787,ZONE IV,2023,8,0,1.296105,Low,Low
788,ZONE IV,2023,9,0,0.691474,Low,Low
789,ZONE IV,2023,10,0,0.000612,Low,Low
790,ZONE IV,2023,11,0,0.000000,Low,Low
791,ZONE IV,2023,12,0,0.000000,Low,Low



Yearly Report (tail):


,BARANGAY,YEAR,Actual_Cases_Yearly,Predicted_Cases_Yearly,Annual_Risk_Total,Annual_Risk_Peak,Max_Monthly_Predicted
61,SANTO NIÑO II,2023,1,2.967507,Low,Low,1.410982
62,VICTORIA REYES,2023,2,2.856110,Low,Low,1.410982
63,ZONE I-B,2023,4,5.667796,Low,Low,1.972870
64,ZONE III,2023,1,2.090772,Low,Low,1.410982
65,ZONE IV,2023,1,3.046594,Low,Low,1.296105


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
# Regression metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred_cases))
mae = mean_absolute_error(y_test, y_pred_cases)
print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}")

# Classification metrics
print("\nMonthly Risk Classification Report:")
print(classification_report(y_true_risk, y_pred_risk))

RMSE: 0.76, MAE: 0.41

Monthly Risk Classification Report:
              precision    recall  f1-score   support

        High       0.00      0.00      0.00         1
         Low       1.00      0.99      1.00       786
      Medium       0.25      0.40      0.31         5

    accuracy                           0.99       792
   macro avg       0.42      0.46      0.43       792
weighted avg       0.99      0.99      0.99       792



In [ ]:
import pickle

# Export the trained XGBoost model to a pickle file
filename = 'xgboost_model.pkl'
pickle.dump(xgb_reg, open(filename, 'wb'))

print(f"XGBoost model exported to {filename}")

# Load the model from the pickle file
loaded_model = pickle.load(open(filename, 'rb'))

# Use the loaded model to make predictions on the test data
y_pred_loaded = loaded_model.predict(X_test)

# Evaluate the loaded model's predictions (should be the same as before)
mse_loaded = mean_squared_error(y_test, y_pred_loaded)
rmse_loaded = np.sqrt(mse_loaded)

print(f"\nRMSE on Test Set using loaded model: {rmse_loaded}")

# You can now use 'loaded_model' for future predictions
# Example: Predict risk levels using the loaded model
y_pred_risk_loaded = [assign_risk_level(c) for c in y_pred_loaded]

print("\nPredicted Risk Levels using loaded model (tail):")
print(y_pred_risk_loaded[-5:])

XGBoost model exported to xgboost_model.pkl

RMSE on Test Set using loaded model: 0.7564539185183903

Predicted Risk Levels using loaded model (tail):
['Low', 'Low', 'Low', 'Low', 'Low']


# Saving and Implementing Model

Save model and threshold

In [ ]:
import joblib
import json

# 1. Save the trained XGBoost regressor
joblib.dump(xgb_reg, '/content/drive/MyDrive/Thesis Data/dengue_xgb_model.pkl')

# 2. Save risk thresholds as JSON (easy to edit later)
risk_config = {
    "monthly": {
        "low_max": 5,
        "medium_max": 10
    },
    "yearly": {
        "low_max": 16,
        "medium_max": 35
    }
}

with open('/content/drive/MyDrive/Thesis Data/risk_thresholds.json', 'w') as f:
    json.dump(risk_config, f, indent=4)

print("✅ Model and thresholds saved!")

How to Load & Use for New Predictions

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json

# ----------------------------
# 1. LOAD SAVED ASSETS
# ----------------------------
# Load model
model = joblib.load('/content/drive/MyDrive/Thesis Data/dengue_xgb_model.pkl')

# Load thresholds
with open('/content/drive/MyDrive/Thesis Data/risk_thresholds.json', 'r') as f:
    risk_config = json.load(f)

# ----------------------------
# 2. DEFINE RISK FUNCTIONS
# ----------------------------
def assign_monthly_risk(cases, config):
    if cases <= config['monthly']['low_max']:
        return 'Low'
    elif cases <= config['monthly']['medium_max']:
        return 'Medium'
    else:
        return 'High'

def assign_yearly_risk(cases, config):
    if cases <= config['yearly']['low_max']:
        return 'Low'
    elif cases <= config['yearly']['medium_max']:
        return 'Medium'
    else:
        return 'High'

# ----------------------------
# 3. PREPARE NEW DATA (EXAMPLE)
# ----------------------------
# ⚠️ IMPORTANT: New data MUST have the SAME columns as training (except BARANGAY, Cases, DATE)
# Example: Predict for January 2024 in "BARANGAY A"

new_data = pd.DataFrame({
    'YEAR': [2024],
    'MONTH': [1],
    'TEMP': [28.5],          # ← Replace with your actual features
    'RAIN': [120.3],
    'HUMIDITY': [75.2],
    'Cases_Lagged': [3],     # Last month's cases (Dec 2023)
    'Cases_Rolling_Avg_3M': [2.7],  # Rolling avg of Oct-Dec 2023
    # ... ADD ALL OTHER FEATURES USED IN TRAINING ...
})

# Ensure column order matches training data
# (Get feature order from your original training set)
feature_cols = [col for col in merged.columns if col not in ['BARANGAY', 'Cases', 'DATE']]
new_data = new_data[feature_cols]  # Reorder to match

# ----------------------------
# 4. MAKE PREDICTION
# ----------------------------
# Predict case count
predicted_cases = model.predict(new_data)[0]
predicted_cases = max(0, predicted_cases)  # Ensure non-negative

# Assign risk
monthly_risk = assign_monthly_risk(predicted_cases, risk_config)

# Output
print(f"Predicted Cases: {predicted_cases:.2f}")
print(f"Monthly Risk Level: {monthly_risk}")

In [ ]:
#check all columns of X_train
print(X_train.columns)

Index(['YEAR', 'MONTH', 'RAINFALL', 'TMAX', 'TMIN', 'WIND_SPEED',
       'WIND_DIRECTION', 'RH', 'TAVG', 'Cases_Lagged', 'Cases_Rolling_Avg_3M'],
      dtype='object')
